# Variational Posterior Approximation

This demo studies posterior approximation for a two-dimensional Bayesian inverse problem arising from a biochemical oxygen demand (BOD) model. The observation is fixed, so every method below approximates the same posterior density $\pi^y(u)$: MAP estimation gives a point summary, Gaussian variational inference gives a restricted distributional approximation, and a RealNVP transport map gives a more flexible learned sampler.

The goal is not only to run the algorithms, but also to keep track of the exact probability densities they optimize. In particular, the posterior is only known up to the evidence $Z(y)$, and all objectives below are written so that constants independent of the optimization variable are handled explicitly.

See Chapter 1, Section 1.2.1: Maximum A Posteriori Estimator; Chapter 2, Section 2.1: Variational Formulation Of Bayes Theorem; Chapter 2, Section 2.2: Canonical Approaches To Variational Inference; Chapter 4: Transport To The Posterior; and Chapter 13, Section 13.3: Normalizing Flows.


## BOD Bayesian inverse problem

We start from the finite-dimensional inverse problem
$$
y = G(u)+\eta, \qquad u\in\mathbb{R}^d, \qquad y\in\mathbb{R}^k,
$$
where $G:\mathbb{R}^d\to\mathbb{R}^k$ is the observation operator and the observational noise is Gaussian,
$$
\eta\sim\mathcal{N}(0,\Gamma).
$$
If the prior density of $u$ is $\rho(u)$, then Bayes' formula gives
$$
\pi^y(u)=\frac{1}{Z(y)}\ell(y\mid u)\rho(u),
\qquad
Z(y)=\int_{\mathbb{R}^d}\ell(y\mid u)\rho(u)\,du.
$$
Here $\ell(y\mid u)$ is the likelihood and $Z(y)$ is the evidence or normalizing constant. For Gaussian noise,
$$
\ell(y\mid u)
= \frac{1}{(2\pi)^{k/2}|\Gamma|^{1/2}}
\exp\left(-\frac12\|y-G(u)\|_{\Gamma^{-1}}^2\right),
$$
where
$$
\|v\|_{\Gamma^{-1}}^2 = v^\top\Gamma^{-1}v.
$$
The evidence $Z(y)$ is usually unavailable in closed form, but it is independent of $u$. This is why MAP estimation and variational objectives can use the unnormalized density $\ell(y\mid u)\rho(u)$.

In the BOD example the latent unknown is
$$
u=(u_1,u_2)^\top\in\mathbb{R}^2,
\qquad
\rho(u)=\mathcal{N}(u;0,I_2).
$$
The physical parameters are an amplitude $A$ and a decay rate $B$. They are constrained through the probability integral transform. If $U\sim\mathcal{N}(0,1)$ and $\Phi$ is the standard normal CDF, then $\Phi(U)\sim\mathrm{Uniform}(0,1)$. Therefore
$$
A(u_1)=0.4+0.8\Phi(u_1),
\qquad
B(u_2)=0.01+0.3\Phi(u_2)
$$
imply the induced physical priors
$$
A\sim\mathrm{Uniform}(0.4,1.2),
\qquad
B\sim\mathrm{Uniform}(0.01,0.31).
$$
This parameterization lets the algorithms optimize over unconstrained Gaussian coordinates $u$, while the transformed physical parameters remain in their admissible intervals.

For observation times $t\in\{1,2,3,4,5\}$, the biochemical oxygen demand curve is
$$
\mathfrak{B}(t;u)=A(u_1)\left(1-\exp[-B(u_2)t]\right).
$$
The forward operator $G:\mathbb{R}^2\to\mathbb{R}^5$ stacks this curve at the five observation times:
$$
G(u)=
\begin{bmatrix}
\mathfrak{B}(1;u)\\
\mathfrak{B}(2;u)\\
\mathfrak{B}(3;u)\\
\mathfrak{B}(4;u)\\
\mathfrak{B}(5;u)
\end{bmatrix}
=
\begin{bmatrix}
A(u_1)(1-e^{-B(u_2)})\\
A(u_1)(1-e^{-2B(u_2)})\\
A(u_1)(1-e^{-3B(u_2)})\\
A(u_1)(1-e^{-4B(u_2)})\\
A(u_1)(1-e^{-5B(u_2)})
\end{bmatrix}.
$$
The noise covariance is
$$
\Gamma = 10^{-3}I_5.
$$
Thus the posterior density for this demo is
$$
\pi^y(u)
=\frac{1}{Z(y)}
\exp\left[-\frac12\|y-G(u)\|_{\Gamma^{-1}}^2\right]
\rho(u)
\times \frac{1}{(2\pi)^{5/2}|\Gamma|^{1/2}},
$$
or, equivalently up to a $u$-independent constant,
$$
\log\pi^y(u)
= -\frac12\|y-G(u)\|_{\Gamma^{-1}}^2 -\frac12\|u\|_2^2 + C(y).
$$
All experiments use the fixed observation
$$
y=(0.1615,0.1868,0.3949,0.3728,0.4177)^\top.
$$
The orange contours show this unnormalized posterior on a grid. The grid is only for visualization; the optimization and sampling methods operate directly in $u$-space.


## MAP estimation

The maximum a posteriori estimator is the posterior mode,
$$
u_{\mathrm{MAP}}=\operatorname*{argmax}_{u\in\mathbb{R}^2}\pi^y(u).
$$
Using Bayes' formula,
$$
\pi^y(u)=\frac{1}{Z(y)}\ell(y\mid u)\rho(u),
$$
we can take the negative logarithm and discard terms that do not depend on $u$. The evidence $Z(y)$, the Gaussian likelihood normalization $(2\pi)^{-5/2}|\Gamma|^{-1/2}$, and the Gaussian prior normalization $(2\pi)^{-1}$ do not change the optimizer. Hence
$$
\begin{aligned}
u_{\mathrm{MAP}}
&=\operatorname*{argmin}_{u}\left[-\log \ell(y\mid u)-\log\rho(u)\right]\\
&=\operatorname*{argmin}_{u}\left[
\frac12\|y-G(u)\|_{\Gamma^{-1}}^2
+\frac12\|u\|_2^2
\right].
\end{aligned}
$$
The MAP objective used in the code is therefore
$$
\mathcal{L}_{\mathrm{MAP}}(u)
=\frac12\|y-G(u)\|_{\Gamma^{-1}}^2+\frac12\|u\|_2^2.
$$
Since $\Gamma=10^{-3}I_5$, the data-misfit term can also be written as
$$
\frac12\|y-G(u)\|_{\Gamma^{-1}}^2
=\frac{1}{2\times10^{-3}}\sum_{j=1}^5\left(y_j-G(u)_j\right)^2.
$$
The second term, $\frac12\|u\|_2^2$, is the quadratic penalty induced by the standard Gaussian prior in the unconstrained coordinates. The MAP estimate is a useful point summary, but it does not describe posterior spread, skewness, or correlation. The later variational sections approximate the full distribution.

The controls below set the initial point, learning rate, number of gradient steps, and whether to show an animation for the fixed BOD posterior. The displayed defaults match the original interactive notebook: 50 Adam steps with learning rate 0.1, and animation on.

**Parameter exploration.**

- Toggle `force_retrain` only when you want to ignore an existing checkpoint and rerun the MAP optimization with the displayed settings.
- Adjust `u1_init` and `u2_init`. Starts near the posterior ridge usually move quickly to a high-density point; starts farther away may approach a different local region or need more steps.
- Adjust `learning_rate`. Larger values can move rapidly across the contours but may overshoot a narrow high-density region; smaller values give a smoother trajectory but require more steps.
- Adjust `n_steps`. More steps help slow starts reach the high-density region, but once the trajectory has settled the objective curve changes little.

**Recommended test combinations.**

Keep unspecified controls at their displayed defaults.

| `u1_init` | `u2_init` | `learning_rate` | `n_steps` | What to look for |
|---:|---:|---:|---:|---|
| -1.2 | 1.5 | 0.1 | 50 | A smooth path into the main posterior ridge. |
| 1.0 | 0.5 | 0.1 | 50 | A different start that can approach a nearby high-density region. |
| -1.2 | 1.5 | 0.1 | 100 | Slower but steadier progress toward the same posterior region. |


In [ ]:
#@title MAP estimation for the fixed BOD posterior
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributions as dist
import matplotlib.pyplot as plt

from pathlib import Path
import hashlib
import json

def checkpoint_path(method_name, params):
    cache_params = {k: v for k, v in params.items() if k != "force_retrain"}
    encoded = json.dumps(cache_params, sort_keys=True).encode("utf-8")
    cache_key = hashlib.sha256(encoded).hexdigest()[:12]
    cache_dir = Path("checkpoints") / "variational_posterior_approximation" / method_name
    cache_dir.mkdir(parents=True, exist_ok=True)
    return cache_dir / f"{cache_key}.pt"

def load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

FIXED_OBSERVATION = torch.tensor([0.1615, 0.1868, 0.3949, 0.3728, 0.4177], dtype=torch.float32)

class BODPosterior:
    def __init__(self, y, gamma_var=1e-3):
        self.y = torch.as_tensor(y, dtype=torch.float32).flatten()
        self.gamma_var = torch.tensor(float(gamma_var), dtype=torch.float32)
        self.obs_time = torch.arange(1, 6, dtype=torch.float32)
        self.prior = dist.MultivariateNormal(torch.zeros(2), torch.eye(2))
        self.standard_normal = dist.Normal(torch.tensor(0.0), torch.tensor(1.0))

    def physical_parameters(self, u):
        u = torch.atleast_2d(torch.as_tensor(u, dtype=torch.float32))
        cdf_vals = self.standard_normal.cdf(u)
        A = 0.4 + 0.8 * cdf_vals[:, 0]
        B = 0.01 + 0.3 * cdf_vals[:, 1]
        return A, B

    def forward_model(self, u):
        u = torch.atleast_2d(torch.as_tensor(u, dtype=torch.float32))
        A, B = self.physical_parameters(u)
        return A[:, None] * (1.0 - torch.exp(-B[:, None] * self.obs_time[None, :]))

    def log_prob(self, u):
        u = torch.atleast_2d(torch.as_tensor(u, dtype=torch.float32))
        residual = self.y[None, :] - self.forward_model(u)
        log_likelihood = -0.5 * torch.sum(residual * residual, dim=1) / self.gamma_var
        return log_likelihood + self.prior.log_prob(u)

    def potential(self, u):
        return -self.log_prob(torch.atleast_2d(u))[0]

    def grid(self, domain_x1=(-1.5, 1.5), domain_x2=(-0.75, 2.25), n_grid=120):
        x1 = torch.linspace(domain_x1[0], domain_x1[1], n_grid)
        x2 = torch.linspace(domain_x2[0], domain_x2[1], n_grid)
        X1, X2 = torch.meshgrid(x1, x2, indexing="ij")
        points = torch.stack([X1.reshape(-1), X2.reshape(-1)], dim=1)
        with torch.no_grad():
            logp = self.log_prob(points)
            density = torch.exp(logp - logp.max()).reshape(n_grid, n_grid).cpu().numpy()
        return X1.cpu().numpy(), X2.cpu().numpy(), density, points

    def plot_density(self, ax, domain_x1=(-1.5, 1.5), domain_x2=(-0.75, 2.25), n_grid=120, title=None):
        X1, X2, density, _ = self.grid(domain_x1, domain_x2, n_grid)
        ax.contourf(X1, X2, density, levels=35, cmap="Oranges", alpha=0.82)
        ax.contour(X1, X2, density, levels=7, colors="saddlebrown", linewidths=0.5, alpha=0.55)
        ax.set_xlabel("$u_1$")
        ax.set_ylabel("$u_2$")
        ax.set_xlim(domain_x1)
        ax.set_ylim(domain_x2)
        if title is not None:
            ax.set_title(title)

DEFAULTS = {
    "u1_init": -1.2,
    "u2_init": 1.5,
    "learning_rate": 0.1,
    "n_steps": 50,
    "show_animation": True,
    "force_retrain": False,
    "seed": 7,
}

use_default = False #@param {type:"boolean"}
force_retrain = False #@param {type:"boolean"}
u1_init = -1.2 #@param {type:"slider", min:-2.0, max:2.0, step:0.1}
u2_init = 1.5 #@param {type:"slider", min:-2.0, max:2.5, step:0.1}
learning_rate = 0.1 #@param {type:"slider", min:0.01, max:0.20, step:0.01}
n_steps = 50 #@param {type:"slider", min:10, max:200, step:10}
show_animation = True #@param {type:"boolean"}

params = DEFAULTS.copy()
if not use_default:
    params.update({
        "u1_init": float(u1_init),
        "u2_init": float(u2_init),
        "learning_rate": float(learning_rate),
        "n_steps": int(n_steps),
        "show_animation": bool(show_animation),
        "force_retrain": bool(force_retrain),
    })

torch.manual_seed(params["seed"])
np.random.seed(params["seed"])
posterior = BODPosterior(FIXED_OBSERVATION)

def run_map(posterior, u0, learning_rate, n_steps):
    u = torch.tensor(u0, dtype=torch.float32, requires_grad=True)
    optimizer = optim.Adam([u], lr=learning_rate)
    history = []
    losses = []
    for _ in range(n_steps):
        optimizer.zero_grad()
        loss = posterior.potential(u)
        loss.backward()
        optimizer.step()
        history.append(u.detach().clone().numpy())
        losses.append(float(loss.detach()))
    return np.asarray(history), np.asarray(losses)

ckpt_path = checkpoint_path("map_estimation", params)
ckpt = None if params["force_retrain"] or not ckpt_path.exists() else load_checkpoint(ckpt_path)

if ckpt is not None:
    history = ckpt["history"]
    losses = ckpt["losses"]
    print(f"Loaded checkpoint: {ckpt_path}")
else:
    history, losses = run_map(
        posterior,
        [params["u1_init"], params["u2_init"]],
        params["learning_rate"],
        params["n_steps"],
    )
    torch.save({"params": params, "history": history, "losses": losses}, ckpt_path)
    print(f"Saved checkpoint: {ckpt_path}")

milestones = sorted(set([0, max(0, params["n_steps"] // 10 - 1), max(0, params["n_steps"] // 2 - 1), params["n_steps"] - 1]))
fig, axes = plt.subplots(1, len(milestones), figsize=(4.3 * len(milestones), 3.8), constrained_layout=True)
if len(milestones) == 1:
    axes = [axes]
for ax, step in zip(axes, milestones):
    posterior.plot_density(ax, title=f"step {step + 1}")
    ax.plot(history[: step + 1, 0], history[: step + 1, 1], color="black", lw=1.3, alpha=0.8)
    ax.scatter(history[step, 0], history[step, 1], c="royalblue", s=45, edgecolors="white", zorder=5)
plt.show()

fig, ax = plt.subplots(figsize=(5.2, 3.4), constrained_layout=True)
ax.plot(np.arange(1, len(losses) + 1), losses, color="royalblue")
ax.set_xlabel("optimization step")
ax.set_ylabel("negative log posterior, up to a constant")
ax.set_title("MAP objective")
ax.grid(alpha=0.25)
plt.show()

if params["show_animation"]:
    fig_anim, ax_anim = plt.subplots(figsize=(5.2, 4.2))

    def update(frame):
        ax_anim.clear()
        posterior.plot_density(ax_anim, title=f"MAP step {frame + 1}")
        ax_anim.plot(history[: frame + 1, 0], history[: frame + 1, 1], color="black", lw=1.3, alpha=0.8)
        ax_anim.scatter(history[frame, 0], history[frame, 1], c="royalblue", s=45, edgecolors="white", zorder=5)
        return []

    animation = FuncAnimation(fig_anim, update, frames=len(history), interval=80, blit=False)
    plt.close(fig_anim)
    display(HTML(animation.to_jshtml()))

u_map = torch.tensor(history[-1])
A_map, B_map = posterior.physical_parameters(u_map)
print("Fixed observation y =", np.round(FIXED_OBSERVATION.numpy(), 4))
print(f"Final u_MAP = ({u_map[0].item():.3f}, {u_map[1].item():.3f})")
print(f"Physical parameters: A = {A_map.item():.3f}, B = {B_map.item():.3f}")
print(f"Final objective = {losses[-1]:.3f}")


## Gaussian variational inference

Variational inference replaces direct posterior sampling with optimization over a tractable family of densities $q_\theta$. In this section the posterior is still the fixed-observation density $\pi^y(u)$ from the BOD problem. The variational objective is the reverse Kullback-Leibler divergence
$$
D_{\mathrm{KL}}(q_\theta\|\pi^y)
=\int q_\theta(u)\log\frac{q_\theta(u)}{\pi^y(u)}\,du.
$$
Substituting Bayes' formula gives
$$
\begin{aligned}
D_{\mathrm{KL}}(q_\theta\|\pi^y)
&=\mathbb{E}_{u\sim q_\theta}\left[\log q_\theta(u)-\log\ell(y\mid u)-\log\rho(u)+\log Z(y)\right]\\
&=\mathbb{E}_{u\sim q_\theta}\left[\log q_\theta(u)-\log\ell(y\mid u)-\log\rho(u)\right]+\log Z(y).
\end{aligned}
$$
The evidence term $\log Z(y)$ is independent of $\theta$, so minimizing the KL divergence is equivalent to minimizing the negative evidence lower bound,
$$
\mathcal{L}_{\mathrm{VI}}(\theta)
=\mathbb{E}_{u\sim q_\theta}\left[
\log q_\theta(u)-\log\ell(y\mid u)-\log\rho(u)
\right].
$$
Using the MAP loss from the previous section, all $\theta$-independent Gaussian normalization constants can be collected into a constant $C$, giving
$$
\mathcal{L}_{\mathrm{VI}}(\theta)
=\mathbb{E}_{u\sim q_\theta}\left[
\mathcal{L}_{\mathrm{MAP}}(u)+\log q_\theta(u)
\right]+C.
$$
The term $\mathcal{L}_{\mathrm{MAP}}(u)$ attracts samples toward high posterior density, while $\log q_\theta(u)$ penalizes overly diffuse proposals through the entropy contribution. Reverse KL is often mode-seeking: if a family cannot cover all posterior structure, it may prefer a concentrated approximation to spreading mass into low-density regions.

The demo compares two Gaussian variational families,
$$
q_\theta(u)=\mathcal{N}(u;m,\Sigma).
$$
To ensure $\Sigma$ is positive definite, the code parameterizes it by a Cholesky factor $L$ with
$$
\Sigma=LL^\top.
$$
For the mean-field family,
$$
L=\operatorname{diag}(\sigma_1,\sigma_2),
\qquad
\sigma_i>0,
$$
so
$$
\Sigma=\begin{bmatrix}\sigma_1^2&0\\0&\sigma_2^2\end{bmatrix}.
$$
This approximation can change marginal variances but cannot represent posterior correlation between $u_1$ and $u_2$. For the full-covariance Gaussian in two dimensions,
$$
L=\begin{bmatrix}
\ell_{11}&0\\
\ell_{21}&\ell_{22}
\end{bmatrix},
\qquad
\ell_{11}>0,\quad \ell_{22}>0,
$$
and therefore
$$
\Sigma=\begin{bmatrix}
\ell_{11}^2 & \ell_{11}\ell_{21}\\
\ell_{11}\ell_{21} & \ell_{21}^2+\ell_{22}^2
\end{bmatrix}.
$$
The off-diagonal entry lets the ellipses rotate and align with a tilted posterior ridge.

Gradients are estimated with the reparameterization trick. If
$$
\epsilon\sim\mathcal{N}(0,I_2),
$$
then a variational sample can be written as
$$
u=m+L\epsilon.
$$
This expresses the sampling randomness through $\epsilon$, which does not depend on $\theta=(m,L)$. The stochastic objective used by the code is the Monte Carlo estimate
$$
\widehat{\mathcal{L}}_{\mathrm{VI}}(\theta)
=\frac{1}{N}\sum_{i=1}^N
\left[\log q_\theta(u^{(i)})-\log\ell(y\mid u^{(i)})-\log\rho(u^{(i)})\right],
\qquad
u^{(i)}=m+L\epsilon^{(i)}.
$$

The `family` control selects the Gaussian family. The learning rate, epoch count, and batch size set the stochastic optimizer used for the Monte Carlo estimate of the KL objective. The displayed defaults match the original interactive notebook: 100 epochs, batch size 64, learning rate 0.1, and animation on.

**Parameter exploration.**

- Toggle `force_retrain` only when you want to ignore an existing checkpoint and rerun variational training with the displayed settings.
- Compare `Mean-field diagonal` and `Full covariance`. The full covariance family should better align with the tilted posterior ridge, while the mean-field family tends to underestimate correlated uncertainty.
- Adjust `batch_size`. Small batches make the Monte Carlo objective noisier; larger batches produce steadier contours but require more computation per epoch.
- Adjust `learning_rate`. A large learning rate can quickly find the posterior region but may destabilize the Cholesky parameters; a smaller learning rate may need more epochs.

**Recommended test combinations.**

Keep unspecified controls at their displayed defaults.

| `family` | `learning_rate` | `n_epochs` | `batch_size` |
|---|---:|---:|---:|
| Mean-field diagonal | 0.1 | 100 | 64 |
| Full covariance | 0.1 | 100 | 64 |


In [ ]:
#@title Gaussian variational inference
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributions as dist
import matplotlib.pyplot as plt

from pathlib import Path
import hashlib
import json

def checkpoint_path(method_name, params):
    cache_params = {k: v for k, v in params.items() if k != "force_retrain"}
    encoded = json.dumps(cache_params, sort_keys=True).encode("utf-8")
    cache_key = hashlib.sha256(encoded).hexdigest()[:12]
    cache_dir = Path("checkpoints") / "variational_posterior_approximation" / method_name
    cache_dir.mkdir(parents=True, exist_ok=True)
    return cache_dir / f"{cache_key}.pt"

def load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

FIXED_OBSERVATION = torch.tensor([0.1615, 0.1868, 0.3949, 0.3728, 0.4177], dtype=torch.float32)

class BODPosterior:
    def __init__(self, y, gamma_var=1e-3):
        self.y = torch.as_tensor(y, dtype=torch.float32).flatten()
        self.gamma_var = torch.tensor(float(gamma_var), dtype=torch.float32)
        self.obs_time = torch.arange(1, 6, dtype=torch.float32)
        self.prior = dist.MultivariateNormal(torch.zeros(2), torch.eye(2))
        self.standard_normal = dist.Normal(torch.tensor(0.0), torch.tensor(1.0))

    def physical_parameters(self, u):
        u = torch.atleast_2d(torch.as_tensor(u, dtype=torch.float32))
        cdf_vals = self.standard_normal.cdf(u)
        A = 0.4 + 0.8 * cdf_vals[:, 0]
        B = 0.01 + 0.3 * cdf_vals[:, 1]
        return A, B

    def forward_model(self, u):
        u = torch.atleast_2d(torch.as_tensor(u, dtype=torch.float32))
        A, B = self.physical_parameters(u)
        return A[:, None] * (1.0 - torch.exp(-B[:, None] * self.obs_time[None, :]))

    def log_prob(self, u):
        u = torch.atleast_2d(torch.as_tensor(u, dtype=torch.float32))
        residual = self.y[None, :] - self.forward_model(u)
        log_likelihood = -0.5 * torch.sum(residual * residual, dim=1) / self.gamma_var
        return log_likelihood + self.prior.log_prob(u)

    def potential(self, u):
        return -self.log_prob(torch.atleast_2d(u))[0]

    def grid(self, domain_x1=(-1.5, 1.5), domain_x2=(-0.75, 2.25), n_grid=120):
        x1 = torch.linspace(domain_x1[0], domain_x1[1], n_grid)
        x2 = torch.linspace(domain_x2[0], domain_x2[1], n_grid)
        X1, X2 = torch.meshgrid(x1, x2, indexing="ij")
        points = torch.stack([X1.reshape(-1), X2.reshape(-1)], dim=1)
        with torch.no_grad():
            logp = self.log_prob(points)
            density = torch.exp(logp - logp.max()).reshape(n_grid, n_grid).cpu().numpy()
        return X1.cpu().numpy(), X2.cpu().numpy(), density, points

    def plot_density(self, ax, domain_x1=(-1.5, 1.5), domain_x2=(-0.75, 2.25), n_grid=120, title=None):
        X1, X2, density, _ = self.grid(domain_x1, domain_x2, n_grid)
        ax.contourf(X1, X2, density, levels=35, cmap="Oranges", alpha=0.82)
        ax.contour(X1, X2, density, levels=7, colors="saddlebrown", linewidths=0.5, alpha=0.55)
        ax.set_xlabel("$u_1$")
        ax.set_ylabel("$u_2$")
        ax.set_xlim(domain_x1)
        ax.set_ylim(domain_x2)
        if title is not None:
            ax.set_title(title)

DEFAULTS = {
    "family": "Full covariance",
    "u1_init": 0.0,
    "u2_init": 0.0,
    "learning_rate": 0.1,
    "n_epochs": 100,
    "batch_size": 64,
    "show_animation": True,
    "force_retrain": False,
    "seed": 2,
}

use_default = False #@param {type:"boolean"}
force_retrain = False #@param {type:"boolean"}
family = "Full covariance" #@param ["Mean-field diagonal", "Full covariance"]
u1_init = 0.0 #@param {type:"slider", min:-1.5, max:1.5, step:0.1}
u2_init = 0.0 #@param {type:"slider", min:-1.0, max:2.0, step:0.1}
learning_rate = 0.1 #@param {type:"slider", min:0.005, max:0.20, step:0.005}
n_epochs = 100 #@param {type:"slider", min:10, max:500, step:10}
batch_size = 64 #@param {type:"slider", min:16, max:512, step:16}
show_animation = True #@param {type:"boolean"}

params = DEFAULTS.copy()
if not use_default:
    params.update({
        "family": family,
        "u1_init": float(u1_init),
        "u2_init": float(u2_init),
        "learning_rate": float(learning_rate),
        "n_epochs": int(n_epochs),
        "batch_size": int(batch_size),
        "show_animation": bool(show_animation),
        "force_retrain": bool(force_retrain),
    })

torch.manual_seed(params["seed"])
np.random.seed(params["seed"])
posterior = BODPosterior(FIXED_OBSERVATION)
init_mean = torch.tensor([params["u1_init"], params["u2_init"]], dtype=torch.float32)

class MeanFieldGaussian(nn.Module):
    def __init__(self, init_mean):
        super().__init__()
        self.mean = nn.Parameter(init_mean.clone())
        self.log_std = nn.Parameter(torch.zeros(2))

    def distribution(self):
        return dist.MultivariateNormal(self.mean, scale_tril=torch.diag(torch.exp(self.log_std)))

class FullGaussian(nn.Module):
    def __init__(self, init_mean):
        super().__init__()
        self.mean = nn.Parameter(init_mean.clone())
        self.chol_params = nn.Parameter(torch.zeros(3))

    def distribution(self):
        L = torch.zeros(2, 2)
        L[0, 0] = torch.exp(self.chol_params[0])
        L[1, 0] = self.chol_params[1]
        L[1, 1] = torch.exp(self.chol_params[2])
        return dist.MultivariateNormal(self.mean, scale_tril=L)

model = MeanFieldGaussian(init_mean) if params["family"] == "Mean-field diagonal" else FullGaussian(init_mean)
optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"])
losses = []
snapshots = {}
animation_states = []
milestones = sorted(set([0, params["n_epochs"] // 4, params["n_epochs"] // 2, params["n_epochs"] - 1]))
record_freq = max(1, params["n_epochs"] // 25)
X1, X2, _, grid_points = posterior.grid()

ckpt_path = checkpoint_path("gaussian_vi", params)
ckpt = None if params["force_retrain"] or not ckpt_path.exists() else load_checkpoint(ckpt_path)

if ckpt is not None:
    model.load_state_dict(ckpt["model_state"])
    losses = ckpt["losses"]
    snapshots = ckpt["snapshots"]
    animation_states = ckpt["animation_states"]
    print(f"Loaded checkpoint: {ckpt_path}")
else:
    for epoch in range(params["n_epochs"]):
        q = model.distribution()
        samples = q.rsample((params["batch_size"],))
        loss = (q.log_prob(samples) - posterior.log_prob(samples)).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach()))

        if epoch in milestones or (params["show_animation"] and (epoch % record_freq == 0 or epoch == params["n_epochs"] - 1)):
            with torch.no_grad():
                q_now = model.distribution()
                state = {
                    "epoch": epoch,
                    "samples": q_now.sample((params["batch_size"],)).cpu().numpy(),
                    "mean": q_now.mean.detach().cpu().numpy(),
                    "cov": q_now.covariance_matrix.detach().cpu().numpy(),
                }
                if epoch in milestones:
                    snapshots[epoch] = state
                if params["show_animation"] and (epoch % record_freq == 0 or epoch == params["n_epochs"] - 1):
                    q_density = torch.exp(q_now.log_prob(grid_points)).reshape(X1.shape).cpu().numpy()
                    q_density = q_density / max(float(q_density.max()), 1e-12)
                    state["density"] = q_density
                    animation_states.append(state)
    torch.save({"params": params, "model_state": model.state_dict(), "losses": losses, "snapshots": snapshots, "animation_states": animation_states}, ckpt_path)
    print(f"Saved checkpoint: {ckpt_path}")

fig, axes = plt.subplots(1, len(milestones), figsize=(4.3 * len(milestones), 3.8), constrained_layout=True)
if len(milestones) == 1:
    axes = [axes]
for ax, epoch in zip(axes, milestones):
    posterior.plot_density(ax, title=f"epoch {epoch + 1}")
    snap = snapshots[epoch]
    ax.scatter(snap["samples"][:, 0], snap["samples"][:, 1], s=12, c="royalblue", alpha=0.45, edgecolors="none")
    q_dist = dist.MultivariateNormal(torch.tensor(snap["mean"]), covariance_matrix=torch.tensor(snap["cov"]))
    with torch.no_grad():
        q_density = torch.exp(q_dist.log_prob(grid_points)).reshape(X1.shape).cpu().numpy()
        q_density = q_density / max(float(q_density.max()), 1e-12)
    ax.contour(X1, X2, q_density, levels=6, colors="black", linewidths=0.9)
plt.show()

fig, ax = plt.subplots(figsize=(5.2, 3.4), constrained_layout=True)
ax.plot(np.arange(1, len(losses) + 1), losses, color="royalblue")
ax.set_xlabel("epoch")
ax.set_ylabel("Monte Carlo estimate of reverse KL objective")
ax.set_title("Gaussian VI training objective")
ax.grid(alpha=0.25)
plt.show()

if params["show_animation"] and animation_states:
    fig_anim, ax_anim = plt.subplots(figsize=(5.2, 4.2))

    def update(frame):
        state = animation_states[frame]
        ax_anim.clear()
        posterior.plot_density(ax_anim, title=f"Gaussian VI epoch {state['epoch'] + 1}")
        ax_anim.scatter(state["samples"][:, 0], state["samples"][:, 1], s=12, c="royalblue", alpha=0.45, edgecolors="none")
        ax_anim.contour(X1, X2, state["density"], levels=6, colors="black", linewidths=0.9)
        return []

    animation = FuncAnimation(fig_anim, update, frames=len(animation_states), interval=150, blit=False)
    plt.close(fig_anim)
    display(HTML(animation.to_jshtml()))

q_final = model.distribution()
print("Fixed observation y =", np.round(FIXED_OBSERVATION.numpy(), 4))
print(f"Variational family: {params['family']}")
print("Mean:", np.round(q_final.mean.detach().numpy(), 3))
print("Covariance:\n", np.round(q_final.covariance_matrix.detach().numpy(), 3))
print(f"Average final objective over last 20 epochs: {np.mean(losses[-20:]):.3f}")


## Transport-map variational inference

A transport map replaces a fixed Gaussian variational family by a learned deterministic transformation. Let
$$
z\sim\varrho, \qquad \varrho=\mathcal{N}(0,I_2),
$$
and let
$$
T_\theta:\mathbb{R}^2\to\mathbb{R}^2
$$
be an invertible map. The approximate posterior is the push-forward distribution
$$
q_\theta = (T_\theta)_\#\varrho,
$$
meaning that samples are generated by
$$
u=T_\theta(z), \qquad z\sim\varrho.
$$
The density of $q_\theta$ follows from change of variables:
$$
q_\theta(u)
=\varrho(T_\theta^{-1}(u))
\left|\det\nabla_u T_\theta^{-1}(u)\right|.
$$
Equivalently, if $u=T_\theta(z)$, then
$$
\log q_\theta(T_\theta(z))
=\log\varrho(z)-\log\left|\det\nabla_z T_\theta(z)\right|.
$$
Because $q_\theta=(T_\theta)_\#\varrho$, minimizing the posterior-space divergence $D_{\mathrm{KL}}(q_\theta\|\pi^y)$ can also be viewed in the reference space as
$$
\min_\theta D_{\mathrm{KL}}\left(\varrho\|T_\theta^{-1}{}_{\#}\pi^y\right),
$$
where $T_\theta^{-1}{}_{\#}\pi^y$ is the pull-back of the posterior by $T_\theta^{-1}$. This formula is the transport-map version of variational inference: learn a map whose push-forward sends the simple reference law toward the posterior, or equivalently whose inverse pulls the posterior back toward the reference law.

Substituting this expression into the reverse KL objective gives
$$
\begin{aligned}
D_{\mathrm{KL}}(q_\theta\|\pi^y)
&=\mathbb{E}_{z\sim\varrho}\left[
\log q_\theta(T_\theta(z))-\log\pi^y(T_\theta(z))
\right]\\
&=\mathbb{E}_{z\sim\varrho}\left[
\log\varrho(z)-\log\left|\det\nabla_z T_\theta(z)\right|-
\log\ell(y\mid T_\theta(z))-\log\rho(T_\theta(z))
\right]+\log Z(y).
\end{aligned}
$$
The terms $\log\varrho(z)$ and $\log Z(y)$ are independent of $\theta$. Therefore the trainable transport objective is
$$
\mathcal{L}_{\mathrm{Flow}}(\theta)
= -\mathbb{E}_{z\sim\varrho}\left[
\log\rho(T_\theta(z))+
\log\ell(y\mid T_\theta(z))+
\log\left|\det\nabla_z T_\theta(z)\right|
\right].
$$
In terms of the MAP loss, the same objective is
$$
\mathcal{L}_{\mathrm{Flow}}(\theta)
=\mathbb{E}_{z\sim\varrho}\left[
\mathcal{L}_{\mathrm{MAP}}(T_\theta(z))
-\log\left|\det\nabla_z T_\theta(z)\right|
\right]+C.
$$
Given independent samples $z^{(1)},\ldots,z^{(N)}\sim\varrho$, the empirical objective minimized in the code is
$$
\widehat{\mathcal{L}}_{\mathrm{Flow}}(\theta)
=\frac1N\sum_{i=1}^N\left[
\mathcal{L}_{\mathrm{MAP}}(T_\theta(z^{(i)}))
-\log\left|\det\nabla_zT_\theta(z^{(i)})\right|
\right].
$$
The first term moves transported samples into high posterior density. The log-determinant term rewards maps that allocate enough volume to represent posterior uncertainty.

The map $T_\theta$ is a RealNVP normalizing flow, constructed as a composition of $K$ affine coupling layers,
$$
T_\theta=L_K\circ L_{K-1}\circ\cdots\circ L_1.
$$
For a two-dimensional input $z=(z_1,z_2)^\top$, the layers alternate which coordinate is held fixed. A type-1 layer updates $z_2$ using $z_1$:
$$
L^{(1)}(z)=
\begin{bmatrix}
z_1\\
z_2\exp(s(z_1))+t(z_1)
\end{bmatrix}.
$$
A type-2 layer updates $z_1$ using $z_2$:
$$
L^{(2)}(z)=
\begin{bmatrix}
z_1\exp(s(z_2))+t(z_2)\\
z_2
\end{bmatrix}.
$$
Here $s$ and $t$ are neural networks representing scale and translation. With a binary mask $m\in\{0,1\}^2$ and Hadamard product $\odot$, both cases can be written as
$$
L(z;m)=m\odot z+(1-m)\odot\left[z\odot\exp(s(m\odot z))+t(m\odot z)\right].
$$
For $L^{(1)}$, $m=(1,0)^\top$; for $L^{(2)}$, $m=(0,1)^\top$.

The key computational property is triangularity. For the type-1 layer,
$$
\nabla_z L^{(1)}(z)=
\begin{bmatrix}
1&0\\
\frac{\partial}{\partial z_1}\left[z_2\exp(s(z_1))+t(z_1)\right]&\exp(s(z_1))
\end{bmatrix}.
$$
The determinant is the product of the diagonal entries, so
$$
\log\left|\det\nabla_zL^{(1)}(z)\right|=s(z_1).
$$
For the masked layer this becomes
$$
\log\left|\det\nabla_zL(z;m)\right|
=\sum_{j=1}^2(1-m_j)s_j(m\odot z).
$$
For the full composition $T_\theta$, the log determinant is the sum of these layerwise log determinants.

The inverse map is also explicit. For the type-1 layer, if $z'=L^{(1)}(z)$, then
$$
(L^{(1)})^{-1}(z')=
\begin{bmatrix}
z'_1\\
(z'_2-t(z'_1))\exp[-s(z'_1)]
\end{bmatrix}.
$$
The code uses this inverse to evaluate $q_\theta(u)$ on a grid and to check whether transported samples pulled back by $T_\theta^{-1}$ resemble the reference Gaussian.

The architecture is fixed at the original notebook setting: 6 coupling layers with hidden dimension 128. The displayed training defaults match the original interactive controls: 100 epochs, batch size 64, learning rate 0.001, and animation on.

**Parameter exploration.**

- Toggle `force_retrain` only when you want to ignore an existing checkpoint and rerun flow training with the displayed settings.
- Adjust `n_epochs`. Early snapshots often show samples spreading from the reference distribution into the posterior ridge; longer training improves alignment of black flow contours with orange posterior contours.
- Adjust `batch_size`. Larger batches reduce Monte Carlo noise in the objective and contours, at the cost of more computation per epoch.
- Adjust `n_samples`. This changes only the final diagnostic sampling density, not the trained map.

**Recommended test combinations.**

Keep unspecified controls at their displayed defaults.

| `learning_rate` | `n_epochs` | `batch_size` | `n_samples` | What to look for |
|---:|---:|---:|---:|---|
| 0.001 | 100 | 64 | 400 | Fast run showing the basic deformation into the posterior region. |
| 0.001 | 300 | 64 | 400 | Better alignment between black flow contours and orange posterior contours. |
| 0.0005 | 500 | 128 | 800 | Smoother training and a denser final sample diagnostic. |


In [ ]:
#@title Transport-map variational inference with RealNVP
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributions as dist
import matplotlib.pyplot as plt

from pathlib import Path
import hashlib
import json

def checkpoint_path(method_name, params):
    cache_params = {k: v for k, v in params.items() if k != "force_retrain"}
    encoded = json.dumps(cache_params, sort_keys=True).encode("utf-8")
    cache_key = hashlib.sha256(encoded).hexdigest()[:12]
    cache_dir = Path("checkpoints") / "variational_posterior_approximation" / method_name
    cache_dir.mkdir(parents=True, exist_ok=True)
    return cache_dir / f"{cache_key}.pt"

def load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

FIXED_OBSERVATION = torch.tensor([0.1615, 0.1868, 0.3949, 0.3728, 0.4177], dtype=torch.float32)

class BODPosterior:
    def __init__(self, y, gamma_var=1e-3):
        self.y = torch.as_tensor(y, dtype=torch.float32).flatten()
        self.gamma_var = torch.tensor(float(gamma_var), dtype=torch.float32)
        self.obs_time = torch.arange(1, 6, dtype=torch.float32)
        self.prior = dist.MultivariateNormal(torch.zeros(2), torch.eye(2))
        self.standard_normal = dist.Normal(torch.tensor(0.0), torch.tensor(1.0))

    def physical_parameters(self, u):
        u = torch.atleast_2d(torch.as_tensor(u, dtype=torch.float32))
        cdf_vals = self.standard_normal.cdf(u)
        A = 0.4 + 0.8 * cdf_vals[:, 0]
        B = 0.01 + 0.3 * cdf_vals[:, 1]
        return A, B

    def forward_model(self, u):
        u = torch.atleast_2d(torch.as_tensor(u, dtype=torch.float32))
        A, B = self.physical_parameters(u)
        return A[:, None] * (1.0 - torch.exp(-B[:, None] * self.obs_time[None, :]))

    def log_prob(self, u):
        u = torch.atleast_2d(torch.as_tensor(u, dtype=torch.float32))
        residual = self.y[None, :] - self.forward_model(u)
        log_likelihood = -0.5 * torch.sum(residual * residual, dim=1) / self.gamma_var
        return log_likelihood + self.prior.log_prob(u)

    def potential(self, u):
        return -self.log_prob(torch.atleast_2d(u))[0]

    def grid(self, domain_x1=(-1.5, 1.5), domain_x2=(-0.75, 2.25), n_grid=120):
        x1 = torch.linspace(domain_x1[0], domain_x1[1], n_grid)
        x2 = torch.linspace(domain_x2[0], domain_x2[1], n_grid)
        X1, X2 = torch.meshgrid(x1, x2, indexing="ij")
        points = torch.stack([X1.reshape(-1), X2.reshape(-1)], dim=1)
        with torch.no_grad():
            logp = self.log_prob(points)
            density = torch.exp(logp - logp.max()).reshape(n_grid, n_grid).cpu().numpy()
        return X1.cpu().numpy(), X2.cpu().numpy(), density, points

    def plot_density(self, ax, domain_x1=(-1.5, 1.5), domain_x2=(-0.75, 2.25), n_grid=120, title=None):
        X1, X2, density, _ = self.grid(domain_x1, domain_x2, n_grid)
        ax.contourf(X1, X2, density, levels=35, cmap="Oranges", alpha=0.82)
        ax.contour(X1, X2, density, levels=7, colors="saddlebrown", linewidths=0.5, alpha=0.55)
        ax.set_xlabel("$u_1$")
        ax.set_ylabel("$u_2$")
        ax.set_xlim(domain_x1)
        ax.set_ylim(domain_x2)
        if title is not None:
            ax.set_title(title)

DEFAULTS = {
    "n_layers": 6,
    "hidden_dim": 128,
    "learning_rate": 0.001,
    "n_epochs": 100,
    "batch_size": 64,
    "n_samples": 400,
    "show_animation": True,
    "force_retrain": False,
    "seed": 3,
}

use_default = False #@param {type:"boolean"}
force_retrain = False #@param {type:"boolean"}
learning_rate = 0.001 #@param {type:"slider", min:0.0001, max:0.0100, step:0.0001}
n_epochs = 100 #@param {type:"slider", min:50, max:1000, step:50}
batch_size = 64 #@param {type:"slider", min:32, max:256, step:32}
n_samples = 400 #@param {type:"slider", min:100, max:1500, step:100}
show_animation = True #@param {type:"boolean"}

params = DEFAULTS.copy()
if not use_default:
    params.update({
        "learning_rate": float(learning_rate),
        "n_epochs": int(n_epochs),
        "batch_size": int(batch_size),
        "n_samples": int(n_samples),
        "show_animation": bool(show_animation),
        "force_retrain": bool(force_retrain),
    })

torch.manual_seed(params["seed"])
np.random.seed(params["seed"])
posterior = BODPosterior(FIXED_OBSERVATION)

class RealNVP(nn.Module):
    def __init__(self, n_layers, hidden_dim, scale_bound=1.0):
        super().__init__()
        self.ref = dist.MultivariateNormal(torch.zeros(2), torch.eye(2))
        self.scale_bound = scale_bound
        masks = [[0.0, 1.0], [1.0, 0.0]] * (n_layers // 2)
        self.register_buffer("masks", torch.tensor(masks, dtype=torch.float32))

        def make_net(use_tanh=False):
            layers = [nn.Linear(2, hidden_dim), nn.LeakyReLU(), nn.Linear(hidden_dim, hidden_dim), nn.LeakyReLU(), nn.Linear(hidden_dim, 2)]
            if use_tanh:
                layers.append(nn.Tanh())
            return nn.Sequential(*layers)

        self.s = nn.ModuleList([make_net(use_tanh=True) for _ in range(n_layers)])
        self.t = nn.ModuleList([make_net(use_tanh=False) for _ in range(n_layers)])

    def T(self, z):
        log_det = z.new_zeros(z.shape[0])
        x = z
        for i, mask in enumerate(self.masks):
            x_masked = x * mask
            s = self.scale_bound * self.s[i](x_masked) * (1.0 - mask)
            t = self.t[i](x_masked) * (1.0 - mask)
            x = x_masked + (1.0 - mask) * (x * torch.exp(s) + t)
            log_det = log_det + s.sum(dim=1)
        return x, log_det

    def Tinv(self, x):
        log_det = x.new_zeros(x.shape[0])
        z = x
        for i in reversed(range(len(self.s))):
            mask = self.masks[i]
            z_masked = z * mask
            s = self.scale_bound * self.s[i](z_masked) * (1.0 - mask)
            t = self.t[i](z_masked) * (1.0 - mask)
            z = z_masked + (1.0 - mask) * (z - t) * torch.exp(-s)
            log_det = log_det - s.sum(dim=1)
        return z, log_det

    def sample(self, n_samples):
        z = self.ref.sample((n_samples,))
        x, _ = self.T(z)
        return x

    def approximate_log_prob(self, x):
        z, log_det_inv = self.Tinv(x)
        return self.ref.log_prob(z) + log_det_inv

flow = RealNVP(params["n_layers"], params["hidden_dim"])
optimizer = optim.Adam(flow.parameters(), lr=params["learning_rate"])
losses = []
snapshots = {}
animation_states = []
milestones = sorted(set([0, params["n_epochs"] // 4, params["n_epochs"] // 2, params["n_epochs"] - 1]))
record_freq = max(1, params["n_epochs"] // 25)
X1, X2, _, grid_points = posterior.grid()

ckpt_path = checkpoint_path("transport_realnvp_vi", params)
ckpt = None if params["force_retrain"] or not ckpt_path.exists() else load_checkpoint(ckpt_path)

if ckpt is not None:
    flow.load_state_dict(ckpt["model_state"])
    losses = ckpt["losses"]
    snapshots = ckpt["snapshots"]
    animation_states = ckpt["animation_states"]
    print(f"Loaded checkpoint: {ckpt_path}")
else:
    for epoch in range(params["n_epochs"]):
        z = flow.ref.sample((params["batch_size"],))
        x, log_det = flow.T(z)
        loss = -(posterior.log_prob(x) + log_det).mean()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(flow.parameters(), 1.0)
        optimizer.step()
        losses.append(float(loss.detach()))

        if epoch in milestones or (params["show_animation"] and (epoch % record_freq == 0 or epoch == params["n_epochs"] - 1)):
            with torch.no_grad():
                samples = flow.sample(params["batch_size"]).detach().cpu().numpy()
                log_q = flow.approximate_log_prob(grid_points)
                q_density = torch.exp(log_q - log_q.max()).reshape(X1.shape).cpu().numpy()
                state = {"epoch": epoch, "samples": samples, "density": q_density}
                if epoch in milestones:
                    snapshots[epoch] = state
                if params["show_animation"] and (epoch % record_freq == 0 or epoch == params["n_epochs"] - 1):
                    animation_states.append(state)
    torch.save({"params": params, "model_state": flow.state_dict(), "losses": losses, "snapshots": snapshots, "animation_states": animation_states}, ckpt_path)
    print(f"Saved checkpoint: {ckpt_path}")

fig, axes = plt.subplots(1, len(milestones), figsize=(4.3 * len(milestones), 3.8), constrained_layout=True)
if len(milestones) == 1:
    axes = [axes]
for ax, epoch in zip(axes, milestones):
    posterior.plot_density(ax, title=f"epoch {epoch + 1}")
    ax.scatter(snapshots[epoch]["samples"][:, 0], snapshots[epoch]["samples"][:, 1], s=12, c="royalblue", alpha=0.45, edgecolors="none")
    ax.contour(X1, X2, snapshots[epoch]["density"], levels=6, colors="black", linewidths=0.9)
plt.show()

fig, ax = plt.subplots(figsize=(5.2, 3.4), constrained_layout=True)
ax.plot(np.arange(1, len(losses) + 1), losses, color="royalblue")
ax.set_xlabel("epoch")
ax.set_ylabel("transport VI objective")
ax.set_title("RealNVP training objective")
ax.grid(alpha=0.25)
plt.show()

if params["show_animation"] and animation_states:
    fig_anim, ax_anim = plt.subplots(figsize=(5.2, 4.2))

    def update(frame):
        state = animation_states[frame]
        ax_anim.clear()
        posterior.plot_density(ax_anim, title=f"Transport VI epoch {state['epoch'] + 1}")
        ax_anim.scatter(state["samples"][:, 0], state["samples"][:, 1], s=12, c="royalblue", alpha=0.45, edgecolors="none")
        ax_anim.contour(X1, X2, state["density"], levels=6, colors="black", linewidths=0.9)
        return []

    animation = FuncAnimation(fig_anim, update, frames=len(animation_states), interval=150, blit=False)
    plt.close(fig_anim)
    display(HTML(animation.to_jshtml()))

with torch.no_grad():
    u_samples = flow.sample(params["n_samples"])
    z_back, _ = flow.Tinv(u_samples)

fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.2), constrained_layout=True)
posterior.plot_density(axes[0], title="posterior space")
axes[0].scatter(u_samples[:, 0], u_samples[:, 1], s=10, c="royalblue", alpha=0.45, edgecolors="none", label="$T(z)$ samples")
axes[0].legend(loc="upper left")
z1 = torch.linspace(-4.0, 4.0, 120)
z2 = torch.linspace(-4.0, 4.0, 120)
Z1, Z2 = torch.meshgrid(z1, z2, indexing="ij")
zz = torch.stack([Z1.reshape(-1), Z2.reshape(-1)], dim=1)
with torch.no_grad():
    ref_density = torch.exp(flow.ref.log_prob(zz)).reshape(120, 120).cpu().numpy()
axes[1].contourf(Z1.numpy(), Z2.numpy(), ref_density, levels=35, cmap="Greens", alpha=0.85)
axes[1].scatter(z_back[:, 0], z_back[:, 1], s=10, c="royalblue", alpha=0.45, edgecolors="none", label="$T^{-1}(u)$ noise samples")
axes[1].set_xlim(-4, 4)
axes[1].set_ylim(-4, 4)
axes[1].set_xlabel("$z_1$")
axes[1].set_ylabel("$z_2$")
axes[1].set_title("noise space")
axes[1].legend(loc="upper left")
plt.show()

print("Fixed observation y =", np.round(FIXED_OBSERVATION.numpy(), 4))
print(f"Flow architecture: {params['n_layers']} RealNVP layers, hidden dimension {params['hidden_dim']}")
print(f"Average first 20 losses: {np.mean(losses[:20]):.3f}")
print(f"Average final 20 losses: {np.mean(losses[-20:]):.3f}")
print("Sample mean:", np.round(u_samples.mean(dim=0).numpy(), 3))
print("Sample standard deviation:", np.round(u_samples.std(dim=0).numpy(), 3))
